In [ ]:
import os
from pathlib import Path

import autoroot
import hydra
import matplotlib.pyplot as plt
import torch

from src.functionals.vandermonde import *
from src.modules.lit_module import LitModule
from src.utils import initialize_resolvers

initialize_resolvers()


def data_illustration(inputs, outputs, rho=None, dim=0):

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    ax[0].plot(inputs[0, :, dim].numpy())
    ax[0].plot(outputs[0, :, 0].numpy())

    if rho is not None:
        ax[1].plot(rho[dim])

In [ ]:
path = "./backup_data/lf/impulse"
# path = "./logs/train/airy"
# path = "./logs/train/rand"
# path = "./logs/train/exp"
# path = "./logs/train/step"


path = Path(path)
ckpt = max((path / "checkpoints").glob("*.ckpt"), key=os.path.getctime)
ckpt_init = path / "checkpoints" / "initial_checkpoint.ckpt"
model = LitModule.load_from_checkpoint(ckpt)

# model_init, model= LitModule.load_from_checkpoint(ckpt_init),LitModule.load_from_checkpoint(ckpt)
# ssm_init, ssm = model_init.model.layers[0], model.model.layers[0]
ssm = model.model.layers[0]
lf = hydra.utils.instantiate(model.hparams.cfg.data).lf
seq_length, D, dt = lf.T, lf.D, lf.dt
print(lf)
# print(ckpt.name)

### Target

In [ ]:
data1 = lf.dataset_construction()
time_steps = torch.arange(seq_length, dtype=torch.int64) * dt
data_illustration(data1[:][0], data1[:][1], lf.rho_vals.numpy(), dim=0)

## Model

In [ ]:
x = data1[0:20][0]
y = data1[0:20][1]
yhat = model(x.to("cuda:0")).cpu().detach()
# Naive calculate Kernel
# ssm._setup_step()
# plt.plot([torch.sum((ssm.dC*ssm.dA.unsqueeze(1)**i @ ssm.dB)).real.item() for i in range(128)])
K = torch.sum(log_vandermonde(ssm.v, ssm.dtA, seq_length), axis=1).real

data_illustration(x, yhat, rho=K.detach().cpu().numpy(), dim=0)

In [ ]:
def log_vandermonde_naive(v, x, L):
    """
    v: (..., N)
    x: (..., N)
    returns: (..., L) \sum v x^l
    """
    vandermonde_matrix = torch.exp(
        x.unsqueeze(-1) * torch.linspace(0, L, 2 * L).to(x)
    )  # (... N L)
    vandermonde_prod = contract("... n, ... n l -> ... l", v, vandermonde_matrix)  # (... L)
    return vandermonde_prod


plt.plot(
    torch.sum(log_vandermonde_naive(ssm.v, ssm.dtA, seq_length), axis=1)
    .real.cpu()
    .detach()
    .numpy()[0]
)

In [ ]:
plt.plot(sorted(ssm.dt.detach().cpu().numpy()[..., 0]))
plt.plot(ssm_init.dt.detach().cpu().numpy()[..., 0])

In [ ]:
ssm._A - ssm_init._A